# Lab 3: Part-of-Speech Tagger Using Transformer Networks

## Deep Learning and Natural Language Processing

### Objective

This lab implements part-of-speech (POS) tagging using a pretrained Transformer model. The model is trained to assign a grammatical POS label to each word in a Spanish sentence.

The lab includes:

- An explanation of Transformer-based POS tagging
- Loading and preparing a Spanish POS-tagging dataset
- Creating a Hugging Face `DatasetDict`
- Tokenizing words and aligning POS labels with subword tokens
- Fine-tuning a pretrained Transformer for token classification
- Training the model using the Hugging Face `Trainer`
- Evaluating the trained model on the test partition

## Part A: Transformer Networks for Part-of-Speech Tagging

Part-of-speech (POS) tagging is the process of assigning a grammatical category to each word in a sentence, such as noun, verb, adjective, adverb, pronoun, or determiner. Transformer networks are well suited for this task because they analyze each word in relation to the full sentence context rather than processing words independently.

A common approach is to start with a pretrained Transformer language model such as BERT or a multilingual BERT model. The input sentence is first tokenized using the model's tokenizer. Because Transformer tokenizers often divide words into smaller subword units, a single original word may correspond to more than one token. The POS label must therefore be aligned carefully with the tokenized output.

In this lab, the POS tag is assigned only to the first token produced from each original word. Any additional subword tokens are assigned the value `-100`. This value is ignored by the loss function during training, so the model is not penalized for those extra subword positions.

After tokenization, the pretrained Transformer produces a contextualized representation for every token. These representations contain information about both the token itself and the surrounding words. A token-classification layer is placed on top of the Transformer. This layer predicts one POS category for each relevant token.

During training, the predicted POS labels are compared with the correct labels from the training data. The model parameters are updated through backpropagation so that the predictions gradually become more accurate.

For evaluation, the trained model is applied to previously unseen sentences in the test dataset. The predicted POS tags are compared with the true tags using metrics such as accuracy, precision, recall, and F1 score. These metrics indicate how well the Transformer has learned to identify grammatical roles in Spanish text.

## Part B: Model Implementation

### Step 1: Initialize the required libraries

The following code imports the main libraries used in this lab. PyTorch is used for deep learning, NumPy and pandas support data processing, and `tqdm` provides progress bars. Hugging Face libraries will be used later for datasets, tokenization, model training, and evaluation.

In [2]:
import torch
import numpy as np
import pandas as pd

from tqdm.auto import tqdm

print("PyTorch version:", torch.__version__)
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.10.0+cpu
NumPy version: 1.26.4
Pandas version: 2.3.3
CUDA available: False


### Step 2: Read the Words and POS Tags from the Spanish Dataset

The next step is to load the Spanish part-of-speech tagging dataset. Each example contains a sequence of words together with the corresponding POS tags. These word-level labels will later be aligned with the subword tokens generated by the Transformer tokenizer.

In [3]:
import datasets
import transformers

print("Datasets version:", datasets.__version__)
print("Transformers version:", transformers.__version__)

Datasets version: 4.5.0
Transformers version: 5.2.0


In [4]:
%pip install conllu

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
from conllu import parse_incr

def read_tags(filename):
    data = {"words": [], "tags": []}

    with open(filename, encoding="utf-8") as f:
        for sentence in tqdm(parse_incr(f)):
            words = []
            tags = []

            for token in sentence:
                # Keep only actual syntactic words with a valid UPOS tag.
                # Multiword-token rows have IDs such as (30, "-", 31)
                # and do not have their own POS tag.
                if isinstance(token["id"], int) and token["upos"] not in (None, "_"):
                    words.append(token["form"])
                    tags.append(token["upos"])

            data["words"].append(words)
            data["tags"].append(tags)

    return pd.DataFrame(data)

In [6]:
from pathlib import Path
import urllib.request

# Create local dataset folder
data_dir = Path("data/UD_Spanish-AnCora")
data_dir.mkdir(parents=True, exist_ok=True)

base_url = "https://raw.githubusercontent.com/UniversalDependencies/UD_Spanish-AnCora/master/"

files = {
    "train": "es_ancora-ud-train.conllu",
    "validation": "es_ancora-ud-dev.conllu",
    "test": "es_ancora-ud-test.conllu",
}

# Download the files only if they are not already present
for split, filename in files.items():
    file_path = data_dir / filename

    if not file_path.exists():
        print(f"Downloading {filename}...")
        urllib.request.urlretrieve(base_url + filename, file_path)
    else:
        print(f"{filename} already exists.")

print("\nDataset files are ready.")

es_ancora-ud-train.conllu already exists.
es_ancora-ud-dev.conllu already exists.
es_ancora-ud-test.conllu already exists.

Dataset files are ready.


In [7]:
train_df = read_tags(data_dir / files["train"])
valid_df = read_tags(data_dir / files["validation"])
test_df = read_tags(data_dir / files["test"])

print("Training sentences  :", len(train_df))
print("Validation sentences:", len(valid_df))
print("Test sentences      :", len(test_df))

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

Training sentences  : 14287
Validation sentences: 1654
Test sentences      : 1721


In [8]:
print("First training example:\n")

print("Words:")
print(train_df.iloc[0]["words"])

print("\nPOS Tags:")
print(train_df.iloc[0]["tags"])

print("\nNumber of words:", len(train_df.iloc[0]["words"]))
print("Number of POS tags:", len(train_df.iloc[0]["tags"]))

First training example:

Words:
['Las', 'reservas', 'de', 'oro', 'y', 'divisas', 'de', 'Rusia', 'subieron', '800', 'millones', 'de', 'dólares', 'y', 'el', '26', 'de', 'mayo', 'equivalían', 'a', '19.100', 'millones', 'de', 'dólares', ',', 'informó', 'hoy', 'un', 'comunicado', 'de', 'el', 'Banco', 'Central', '.']

POS Tags:
['DET', 'NOUN', 'ADP', 'NOUN', 'CCONJ', 'NOUN', 'ADP', 'PROPN', 'VERB', 'NUM', 'NOUN', 'ADP', 'NOUN', 'CCONJ', 'DET', 'NUM', 'ADP', 'NOUN', 'VERB', 'ADP', 'NUM', 'NOUN', 'ADP', 'NOUN', 'PUNCT', 'VERB', 'ADV', 'DET', 'NOUN', 'ADP', 'DET', 'PROPN', 'PROPN', 'PUNCT']

Number of words: 34
Number of POS tags: 34


In [9]:
# Inspect all POS tags in the training data
all_train_tags = [
    tag
    for tags in train_df["tags"]
    for tag in tags
]

unique_tags = sorted(set(all_train_tags))

print("Unique POS tags:")
print(unique_tags)

print("\nNumber of unique tags:", len(unique_tags))
print("Number of '_' tags:", all_train_tags.count("_"))

Unique POS tags:
['ADJ', 'ADP', 'ADV', 'AUX', 'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM', 'PRON', 'PROPN', 'PUNCT', 'SCONJ', 'SYM', 'VERB', 'X']

Number of unique tags: 16
Number of '_' tags: 0


### Step 3: Create a Hugging Face DatasetDict

The pandas DataFrames are converted into Hugging Face `Dataset` objects and organized into a `DatasetDict` containing training, validation, and test partitions. This structure makes it easier to apply tokenization and preprocessing consistently to all dataset splits.

In [10]:
from datasets import Dataset, DatasetDict

dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df, preserve_index=False),
    "validation": Dataset.from_pandas(valid_df, preserve_index=False),
    "test": Dataset.from_pandas(test_df, preserve_index=False)
})

print(dataset)
print("\nTraining example:")
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['words', 'tags'],
        num_rows: 14287
    })
    validation: Dataset({
        features: ['words', 'tags'],
        num_rows: 1654
    })
    test: Dataset({
        features: ['words', 'tags'],
        num_rows: 1721
    })
})

Training example:
{'words': ['Las', 'reservas', 'de', 'oro', 'y', 'divisas', 'de', 'Rusia', 'subieron', '800', 'millones', 'de', 'dólares', 'y', 'el', '26', 'de', 'mayo', 'equivalían', 'a', '19.100', 'millones', 'de', 'dólares', ',', 'informó', 'hoy', 'un', 'comunicado', 'de', 'el', 'Banco', 'Central', '.'], 'tags': ['DET', 'NOUN', 'ADP', 'NOUN', 'CCONJ', 'NOUN', 'ADP', 'PROPN', 'VERB', 'NUM', 'NOUN', 'ADP', 'NOUN', 'CCONJ', 'DET', 'NUM', 'ADP', 'NOUN', 'VERB', 'ADP', 'NUM', 'NOUN', 'ADP', 'NOUN', 'PUNCT', 'VERB', 'ADV', 'DET', 'NOUN', 'ADP', 'DET', 'PROPN', 'PROPN', 'PUNCT']}


### Step 4: Tokenize the Text and Align POS Labels

Transformer tokenizers may split a single word into multiple subword tokens. Therefore, the original word-level POS labels must be aligned with the tokenized representation.

In this implementation, the POS label is assigned only to the first token of each original word. Special tokens and additional subword tokens are assigned `-100`, which causes them to be ignored when the training loss is calculated.

In [11]:
# Create POS label mappings

label_list = sorted(unique_tags)

label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

print("Number of POS labels:", len(label_list))
print("\nLabel mappings:")

for label, idx in label2id.items():
    print(f"{idx:2d} -> {label}")

Number of POS labels: 16

Label mappings:
 0 -> ADJ
 1 -> ADP
 2 -> ADV
 3 -> AUX
 4 -> CCONJ
 5 -> DET
 6 -> INTJ
 7 -> NOUN
 8 -> NUM
 9 -> PRON
10 -> PROPN
11 -> PUNCT
12 -> SCONJ
13 -> SYM
14 -> VERB
15 -> X


In [12]:
from transformers import AutoTokenizer

transformer_name = "distilbert/distilbert-base-multilingual-cased"

tokenizer = AutoTokenizer.from_pretrained(transformer_name)

print("Transformer model:", transformer_name)
print("Tokenizer type:", type(tokenizer).__name__)
print("Vocabulary size:", tokenizer.vocab_size)

config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

c:\Users\dubey\Documents\UC_GitLab\.venv_global\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\dubey\.cache\huggingface\hub\models--distilbert--distilbert-base-multilingual-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Transformer model: distilbert/distilbert-base-multilingual-cased
Tokenizer type: BertTokenizer
Vocabulary size: 119547


In [13]:
example_words = dataset["train"][0]["words"]

encoded_example = tokenizer(
    example_words,
    is_split_into_words=True,
    truncation=True
)

tokens = tokenizer.convert_ids_to_tokens(encoded_example["input_ids"])
word_ids = encoded_example.word_ids()

print("Original words:")
print(example_words)

print("\nTokenized output:")
print(tokens)

print("\nWord IDs:")
print(word_ids)

Original words:
['Las', 'reservas', 'de', 'oro', 'y', 'divisas', 'de', 'Rusia', 'subieron', '800', 'millones', 'de', 'dólares', 'y', 'el', '26', 'de', 'mayo', 'equivalían', 'a', '19.100', 'millones', 'de', 'dólares', ',', 'informó', 'hoy', 'un', 'comunicado', 'de', 'el', 'Banco', 'Central', '.']

Tokenized output:
['[CLS]', 'Las', 'reservas', 'de', 'oro', 'y', 'divisa', '##s', 'de', 'Rusia', 'sub', '##ieron', '800', 'millones', 'de', 'dólares', 'y', 'el', '26', 'de', 'mayo', 'e', '##qui', '##val', '##ían', 'a', '19', '.', '100', 'millones', 'de', 'dólares', ',', 'info', '##rmó', 'hoy', 'un', 'comunica', '##do', 'de', 'el', 'Banco', 'Central', '.', '[SEP]']

Word IDs:
[None, 0, 1, 2, 3, 4, 5, 5, 6, 7, 8, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 18, 18, 18, 19, 20, 20, 20, 21, 22, 23, 24, 25, 25, 26, 27, 28, 28, 29, 30, 31, 32, 33, None]


In [14]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["words"],
        truncation=True,
        is_split_into_words=True
    )

    all_labels = []

    for batch_index, tags in enumerate(examples["tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=batch_index)

        labels = []
        previous_word_id = None

        for word_id in word_ids:

            # Special tokens such as <s> and </s>
            if word_id is None:
                labels.append(-100)

            # First token of an original word
            elif word_id != previous_word_id:
                labels.append(label2id[tags[word_id]])

            # Additional subword token from the same word
            else:
                labels.append(-100)

            previous_word_id = word_id

        all_labels.append(labels)

    tokenized_inputs["labels"] = all_labels

    return tokenized_inputs


tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True
)

print(tokenized_dataset)

Map:   0%|          | 0/14287 [00:00<?, ? examples/s]

Map:   0%|          | 0/1654 [00:00<?, ? examples/s]

Map:   0%|          | 0/1721 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['words', 'tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 14287
    })
    validation: Dataset({
        features: ['words', 'tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 1654
    })
    test: Dataset({
        features: ['words', 'tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 1721
    })
})


In [15]:
example = tokenized_dataset["train"][0]

tokens = tokenizer.convert_ids_to_tokens(example["input_ids"])
labels = example["labels"]

print(f"{'TOKEN':20} {'LABEL ID':10} {'POS TAG'}")
print("-" * 45)

for token, label_id in zip(tokens, labels):
    if label_id == -100:
        pos_tag = "IGNORED"
    else:
        pos_tag = id2label[label_id]

    print(f"{token:20} {label_id:<10} {pos_tag}")

TOKEN                LABEL ID   POS TAG
---------------------------------------------
[CLS]                -100       IGNORED
Las                  5          DET
reservas             7          NOUN
de                   1          ADP
oro                  7          NOUN
y                    4          CCONJ
divisa               7          NOUN
##s                  -100       IGNORED
de                   1          ADP
Rusia                10         PROPN
sub                  14         VERB
##ieron              -100       IGNORED
800                  8          NUM
millones             7          NOUN
de                   1          ADP
dólares              7          NOUN
y                    4          CCONJ
el                   5          DET
26                   8          NUM
de                   1          ADP
mayo                 7          NOUN
e                    14         VERB
##qui                -100       IGNORED
##val                -100       IGNORED
##ían           

### Step 5: Create the Transformer Model

A pretrained XLM-RoBERTa model is adapted for token classification. The original Transformer provides contextual representations for each token, while a classification layer predicts one of the 16 POS tags for each labeled token.

The mappings between label names and numeric IDs are supplied to the model so that predictions can be converted back into readable POS tags.

In [16]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    transformer_name,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

print("Model:", model.__class__.__name__)
print("Number of POS labels:", model.config.num_labels)
print("Classifier output size:", model.classifier.out_features)

model.safetensors:   0%|          | 0.00/542M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert/distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model: DistilBertForTokenClassification
Number of POS labels: 16
Classifier output size: 16


### Step 6: Create the Trainer Object and Train the Model

The Hugging Face `Trainer` API is used to fine-tune the pretrained DistilBERT model on the Spanish POS-tagging dataset. A token-classification data collator dynamically pads each batch, while the evaluation function calculates performance using only valid POS labels and ignores positions marked with `-100`.

Because this experiment is being run on a CPU-only system, a relatively small batch size and a limited number of training epochs are used to keep training computationally practical.

In [17]:
from transformers import DataCollatorForTokenClassification
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []

    for prediction, label in zip(predictions, labels):
        for pred_id, label_id in zip(prediction, label):
            if label_id != -100:
                true_predictions.append(pred_id)
                true_labels.append(label_id)

    accuracy = accuracy_score(true_labels, true_predictions)

    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels,
        true_predictions,
        average="weighted",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

print("Data collator:", type(data_collator).__name__)
print("Evaluation metrics are ready.")

Data collator: DataCollatorForTokenClassification
Evaluation metrics are ready.


In [18]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./pos_tagger_results",

    learning_rate=2e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    num_train_epochs=1,

    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="no",

    logging_strategy="steps",
    logging_steps=200,

    report_to="none",

    use_cpu=True
)

print("Training arguments created successfully.")
print("Epochs:", training_args.num_train_epochs)
print("Train batch size:", training_args.per_device_train_batch_size)
print("Evaluation strategy:", training_args.eval_strategy)

Training arguments created successfully.
Epochs: 1
Train batch size: 8
Evaluation strategy: IntervalStrategy.EPOCH


In [19]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Trainer created successfully.")
print("Training examples:", len(tokenized_dataset["train"]))
print("Validation examples:", len(tokenized_dataset["validation"]))

Trainer created successfully.
Training examples: 14287
Validation examples: 1654


In [20]:
train_result = trainer.train()

print("\nTraining completed.")
print(train_result)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.055880,0.051593,0.986573,0.986521,0.986573,0.986535



Training completed.
TrainOutput(global_step=1786, training_loss=0.11963009967782597, metrics={'train_runtime': 3218.1438, 'train_samples_per_second': 4.44, 'train_steps_per_second': 0.555, 'total_flos': 277916654350944.0, 'train_loss': 0.11963009967782597, 'epoch': 1.0})


### Step 7: Evaluate the Testing Partition

After training, the final model is evaluated on the independent test partition. The test data was not used to update the model parameters during training. Therefore, performance on this partition provides a better indication of how well the trained POS tagger generalizes to unseen Spanish sentences.

Accuracy measures the percentage of correctly predicted POS tags. Precision measures how often predicted POS labels are correct, recall measures how well the model identifies the true POS labels, and the F1 score provides a balance between precision and recall.

In [21]:
test_results = trainer.evaluate(
    eval_dataset=tokenized_dataset["test"],
    metric_key_prefix="test"
)

print("Test Results")
print("-" * 40)

for metric, value in test_results.items():
    if isinstance(value, float):
        print(f"{metric:25s}: {value:.6f}")
    else:
        print(f"{metric:25s}: {value}")

Test Results
----------------------------------------
test_loss                : 0.053345
test_accuracy            : 0.985920
test_precision           : 0.985898
test_recall              : 0.985920
test_f1                  : 0.985843
test_runtime             : 66.802000
test_samples_per_second  : 25.763000
test_steps_per_second    : 3.233000
epoch                    : 1.000000


## Part C: Analysis

### Explanation of the Commands Used in Part B

The implementation begins by importing PyTorch, NumPy, pandas, and tqdm. PyTorch provides the deep-learning framework, NumPy and pandas support data processing, and tqdm displays progress during dataset processing.

The Spanish AnCora dataset is read from CoNLL-U files. Each sentence is converted into a list of words and a corresponding list of Universal Part-of-Speech (UPOS) tags. Multiword-token entries without valid POS labels are excluded so that every retained word has one valid POS tag.

The training, validation, and test DataFrames are then converted into Hugging Face `Dataset` objects and combined into a `DatasetDict`. This allows the same preprocessing operations to be applied consistently to all three partitions.

The `distilbert-base-multilingual-cased` tokenizer converts the Spanish words into Transformer tokens. Some words are divided into multiple subword tokens. The `word_ids()` method is used to determine which subword corresponds to each original word. The POS label is assigned only to the first subword of each word. Special tokens and additional subwords receive a label of `-100`, which causes those positions to be ignored when the training loss is calculated.

A pretrained multilingual DistilBERT model is loaded using `AutoModelForTokenClassification`. The model is configured with 16 output labels corresponding to the 16 POS categories present in the dataset. The final classification layer is trained specifically for the POS-tagging task.

The Hugging Face `Trainer` manages batching, forward and backward propagation, parameter updates, validation, and evaluation. A learning rate of `2e-5`, batch size of 8, and one training epoch were used. Because the experiment was performed on a CPU-only system, one epoch was selected to keep the training computationally practical.

### Evaluation Results

After one training epoch, the model achieved a validation accuracy of approximately **98.66%** and a validation F1 score of approximately **98.65%**.

On the independent test partition, the model obtained:

- **Test loss:** 0.053345
- **Accuracy:** 0.985920, or approximately **98.59%**
- **Precision:** 0.985898, or approximately **98.59%**
- **Recall:** 0.985920, or approximately **98.59%**
- **F1 score:** 0.985843, or approximately **98.58%**

These results indicate that the Transformer model learned the grammatical patterns in the Spanish training data effectively. The accuracy shows that approximately 98.6% of the evaluated word-level POS labels were predicted correctly. Precision and recall are also very close to the accuracy, showing balanced performance across the POS predictions.

The test performance is only slightly lower than the validation performance. Validation accuracy was approximately 98.66%, compared with test accuracy of approximately 98.59%. This small difference suggests that the model generalized effectively to previously unseen sentences and showed little evidence of overfitting after one epoch.

Overall, the experiment demonstrates that fine-tuning a pretrained multilingual Transformer can produce highly accurate POS tagging even with a relatively short fine-tuning process.

## References

Hugging Face. (n.d.). *Transformers documentation*. https://huggingface.co/docs/transformers/

Universal Dependencies. (n.d.). *Spanish AnCora*. https://universaldependencies.org/treebanks/es_ancora/

Wolf, T., Debut, L., Sanh, V., Chaumond, J., Delangue, C., Moi, A., Cistac, P., Rault, T., Louf, R., Funtowicz, M., Davison, J., Shleifer, S., von Platen, P., Ma, C., Jernite, Y., Plu, J., Xu, C., Le Scao, T., Gugger, S., Drame, M., Lhoest, Q., & Rush, A. M. (2020). Transformers: State-of-the-art natural language processing. *Proceedings of the 2020 Conference on Empirical Methods in Natural Language Processing: System Demonstrations*, 38–45.

## AI Use Disclosure

ChatGPT was used to support understanding of Transformer-based part-of-speech tagging, assist with organizing the notebook, explain the implementation steps, and help interpret the model evaluation results. All code was executed by the author in the local development environment. Dataset processing, training outputs, and evaluation metrics were reviewed and verified against the actual notebook results and relevant Hugging Face and Universal Dependencies documentation.